In [9]:
#! pip install kafka-python

In [10]:
from kafka import KafkaAdminClient

In [11]:
# create connection to kafka broker 

admin= KafkaAdminClient(
    bootstrap_servers="localhost:9092"
)
print("connected successfully")

connected successfully


In [12]:
#list all topics 
topics = admin.list_topics()
print(topics)

[]


In [13]:
admin.describe_cluster()

{'throttle_time_ms': 0,
 'brokers': [{'node_id': 1, 'host': 'localhost', 'port': 9092, 'rack': None}],
 'cluster_id': '5L6g3nShT-eMCtK--X86sw',
 'controller_id': 1,
 'authorized_operations': ['CREATE',
  'ALTER',
  'DESCRIBE',
  'CLUSTER_ACTION',
  'DESCRIBE_CONFIGS',
  'ALTER_CONFIGS',
  'IDEMPOTENT_WRITE']}

In [14]:
from kafka.admin import NewTopic

In [15]:
# define topic configuration 
sensor_topic = NewTopic(
    name= "sensor-telemetry",
    num_partitions=3,
    replication_factor=1
)

In [ ]:
# create  topic in kafka 
admin.create_topics(
    new_topics=[sensor_topic],
    validate_only=False
)

print("topics created sucessfully")

topics crreated sucessfully


In [17]:
# verify topic exists  or not
topics =admin.list_topics()
print(topics)

['sensor-telemetry']


In [18]:
#inspect topic metadata
from kafka import KafkaConsumer
consumer = KafkaConsumer(
    bootstrap_servers= "localhost:9092"
)

partitions= consumer.partitions_for_topic(
    "sensor-telemetry"
)
print(partitions)

{0, 1, 2}


In [19]:
# import kafka
from kafka import KafkaProducer
import json

In [20]:
#create producer connection
producer= KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer= lambda v: json.dumps(v).encode("utf-8")
)

In [40]:
# create sensor event 

sensor_event= {
    "sensor_id": "S101",
    "temprature": 24.5,
    "humidity": 60
}

print(sensor_event)

{'sensor_id': 'S101', 'temprature': 24.5, 'humidity': 60}


In [51]:
# send message to kafka 
future = producer.send(
    "sensor-telemetry",
    key=b"sensor_id",
    value= sensor_event
)

print("message sent")

message sent


In [57]:
# wait for kafka confirmation
record_metadata= future.get(timeout=10)   #producer wait for 10 sec
print(record_metadata)

RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=2, timestamp=1781713787585, checksum=None, serialized_key_size=9, serialized_value_size=336, serialized_header_size=-1)


In [59]:
sensor_event= [
{"sensor_id": "S101","temprature": 24},
{"sensor_id": "S102","temprature": 26},
{"sensor_id": "S103","temprature": 45},
{"sensor_id": "S104","temprature": 70},
{"sensor_id": "S105","temprature": 24},
{"sensor_id": "S106","temprature": 26},
{"sensor_id": "S107","temprature": 45},
{"sensor_id": "S108","temprature": 70}   
]

for event in sensor_event:
    metadata=producer.send(
        "sensor-elementary",
        key= event["sensor_id"].encode("utf-8"),
        value=event

    ).get()

    print(
        f"partition={metadata.partition}",
        f"offset={metadata.offset}"
    )

partition=0 offset=80
partition=0 offset=81
partition=0 offset=82
partition=0 offset=83
partition=0 offset=84
partition=0 offset=85
partition=0 offset=86
partition=0 offset=87


In [60]:
producer.flush()
print("all message sent")

all message sent


In [61]:
#create our first consumer
from kafka import kafkaConsumer

ImportError: cannot import name 'kafkaConsumer' from 'kafka' (c:\Users\himal\Data__Engineering\venv\Lib\site-packages\kafka\__init__.py)

In [ ]:
import json
consumer= kafkaConsumer(
    "sensor-telemetry",
    bootstrap_Servers="localhost:9092",
    auto_offset_reset="earliest",
    value_deserialize= lambda m:josn.loads(m.decode("utf-8")),
    consumer_timeout_ms=500
    )